# BERTopic — Basic Introduction

Companion notebook for the **Topic Modeling & BERTopic** lesson of the Insper Machine Learning course.

The pipeline: **sentence embeddings → UMAP → HDBSCAN → c-TF-IDF** — every component is a technique covered in Part II of the course.

> Runs fine on CPU; a GPU speeds up the embedding step. In Google Colab: `Runtime > Change runtime type > GPU`.

In [ ]:
%pip install -q bertopic

## 1. Load a corpus

The classic 20 Newsgroups dataset: ~18k forum posts on 20 themes (sports, religion, cryptography, cars...). We strip headers/footers so the model sees only the text.

In [ ]:
from sklearn.datasets import fetch_20newsgroups

docs = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes')).data
print(len(docs), 'documents')
print(docs[0][:300])

## 2. Fit BERTopic

With defaults: `all-MiniLM-L6-v2` sentence embeddings, UMAP to 5 dimensions, HDBSCAN clustering, c-TF-IDF topic descriptions. We fix UMAP's `random_state` for reproducible topics.

In [ ]:
from bertopic import BERTopic
from umap import UMAP

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                  metric='cosine', random_state=42)

topic_model = BERTopic(umap_model=umap_model, min_topic_size=30, verbose=True)
topics, probs = topic_model.fit_transform(docs)

## 3. Inspect the discovered topics

Topic `-1` collects the outliers — documents HDBSCAN placed in no dense region. The number of topics was **not** chosen by us; it emerged from the data.

In [ ]:
topic_model.get_topic_info().head(15)

In [ ]:
# Top c-TF-IDF words of one topic
topic_model.get_topic(0)

In [ ]:
# Semantic topic search — no keyword match needed
topic_model.find_topics('space exploration', top_n=3)

## 4. Visualizations

Interactive Plotly figures — hover to explore.

In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_barchart(top_n_topics=12)

In [ ]:
topic_model.visualize_heatmap(top_n_topics=20)

## 5. Assign topics to new documents

In [ ]:
new_docs = [
    'My car needs new brake pads and an oil change',
    'The goalie made an incredible save in overtime',
    'Public key cryptography relies on hard math problems',
]
new_topics, new_probs = topic_model.transform(new_docs)
for doc, t in zip(new_docs, new_topics):
    words = [w for w, _ in topic_model.get_topic(t)][:5] if t != -1 else ['<outlier>']
    print(f'{t:>4}  {words}  <-  {doc}')

## 6. Experiments to try

1. **Granularity** — refit with `min_topic_size=100`. How does the number of topics change?
2. **Outliers** — use `topic_model.reduce_outliers(docs, topics)` and compare `get_topic_info()`.
3. **Better labels** — pass `vectorizer_model=CountVectorizer(stop_words='english', ngram_range=(1, 2))` and compare topic keywords.
4. **Merge topics** — `topic_model.reduce_topics(docs, nr_topics=20)`.
5. **Portuguese** — build a small corpus of PT-BR sentences and fit `BERTopic(language='multilingual')`.